In [ ]:
import numpy as np

# -----------------------------
# 1. Sentence
# -----------------------------
sentence = "the cat sat on the mat".split()

# Vocabulary
vocab = list(set(sentence))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}

vocab_size = len(vocab)

# -----------------------------
# 2. Create Skip-Gram pairs
# -----------------------------
window_size = 1
pairs = []

for i, center_word in enumerate(sentence):
    center_idx = word_to_idx[center_word]
    for j in range(max(0, i - window_size), min(len(sentence), i + window_size + 1)):
        if i != j:
            context_word = sentence[j]
            context_idx = word_to_idx[context_word]
            pairs.append((center_idx, context_idx))

# -----------------------------
# 3. One-hot encoding
# -----------------------------
def one_hot(idx, size):
    vec = np.zeros(size)
    vec[idx] = 1
    return vec

X = np.array([one_hot(center, vocab_size) for center, _ in pairs])
Y = np.array([one_hot(context, vocab_size) for _, context in pairs])

# -----------------------------
# 4. Initialize weights
# -----------------------------
np.random.seed(42)
embedding_dim = 5

W1 = np.random.randn(vocab_size, embedding_dim)
W2 = np.random.randn(embedding_dim, vocab_size)

# -----------------------------
# 5. Softmax
# -----------------------------
def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

# -----------------------------
# 6. Training
# -----------------------------
learning_rate = 0.05
epochs = 3000

for epoch in range(epochs):
    loss = 0

    for x, y_true in zip(X, Y):
        # Forward
        h = np.dot(x, W1)
        u = np.dot(h, W2)
        y_pred = softmax(u)

        loss += -np.sum(y_true * np.log(y_pred + 1e-9))

        # Backprop
        error = y_pred - y_true

        dW2 = np.outer(h, error)
        dW1 = np.outer(x, np.dot(error, W2.T))

        W2 -= learning_rate * dW2
        W1 -= learning_rate * dW1

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

# -----------------------------
# 7. Predict context words for "sat"
# -----------------------------
sat_idx = word_to_idx["sat"]
sat_vec = one_hot(sat_idx, vocab_size)

h = np.dot(sat_vec, W1)
u = np.dot(h, W2)
probs = softmax(u)

print("\nContext word probabilities for 'sat':\n")
for i, p in sorted(enumerate(probs), key=lambda x: -x[1]):
    print(f"{idx_to_word[i]} : {p:.4f}")

print("\nEmbedding vector for 'sat':")
print(W1[sat_idx])


Epoch 0, Loss: 35.7927
Epoch 500, Loss: 8.0086
Epoch 1000, Loss: 7.8995
Epoch 1500, Loss: 7.8558
Epoch 2000, Loss: 7.8300
Epoch 2500, Loss: 7.8122

Context word probabilities for 'sat':

on : 0.5242
cat : 0.4644
mat : 0.0107
the : 0.0003
sat : 0.0003

Embedding vector for 'sat':
[-0.08583143 -0.15912696 -0.85817144 -0.47391031  1.53154852]
